In [88]:
################### Find consistency pts across 40 years ########

from datetime import datetime, timedelta
import os
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import requests
import sys
import geopandas as gpd
import pandas as pd
from shapely.geometry import MultiPoint
from shapely.geometry import Point
import fiona
import contextily as ctx
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
import simplekml
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from pyproj import Proj, transform
import ast
from pyproj import Transformer
import rasterio
from rasterio.features import geometry_mask
from rasterio.transform import from_origin
from rasterio.warp import calculate_default_transform, reproject
from rasterio.enums import Resampling

from scipy.ndimage import binary_fill_holes
from scipy.ndimage import convolve
from scipy.ndimage import distance_transform_edt
from affine import Affine
from scipy.spatial import KDTree
from rasterio.mask import mask
from rasterio.features import shapes
from shapely.geometry import shape
from simplekml import Kml
from osgeo import gdal
from rasterio.transform import rowcol
import xml.etree.ElementTree as ET
from collections import OrderedDict
import random

In [2]:
def count_non_nan_values(array):
        """Counts non-NaN values in an array while ignoring non-numeric elements."""
        # Filter to keep only numeric data (e.g., ints and floats)
        numeric_values = [x for x in array if isinstance(x, (int, float))] 
        # Convert to NumPy array and count non-NaN values
        return np.sum(~np.isnan(np.array(numeric_values, dtype=float)))
    
def data_prep(path, file, cutoff=0, direction=0, cutlon=0, sensor = 1):
    ####### filter the data with 015042 index ######
    df = pd.read_csv(path + file)
    df['ID'] = range(0, len(df))  # Generates IDs starting from 1
    
    if sensor == 0:
        training_data = df[['lat', 'lon', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'ndmi', 'ndvi', 'savi', 'descriptio']].to_numpy()
    else:
        training_data = df[['lat', 'lon', 'SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7', 'ndmi', 'ndvi', 'savi', 'descriptio']].to_numpy()
    
    predictor = []
    target = []
    coord = []
    # Loop through each (1585, 10) array in training_data
    print('training data is:')
    #print(training_data)
    print(training_data.shape)
    #for array in training_data:
        # Get rows where the target is not 'TZ'
    mask = training_data[:, 11] != 'MZ'
    
    predictor.append(training_data[mask, 2:11])  # First 9 columns
    target.append(training_data[mask, 11])       # Target column
    coord.append(training_data[mask, :2])
    
    # Convert to NumPy arrays if desired
    predictor = np.array(predictor)
    target = np.array(target)
    coord = np.array(coord)
    
    X = predictor.reshape(-1, 9)  # Shape: (5 * 1585, 9)
    y = target.reshape(-1)        # Shape: (5 * 1585,)
    coord = coord.reshape(-1, 2)
    ### cleanse the data by removing nan ####
    valid_indices = []

    # Loop through each row and check for NaN in the first column
    for i in range(X.shape[0]):
        if not np.isnan(X[i, 0]):
            valid_indices.append(i)
    
    # Create new X and y with only valid rows
    X = X[valid_indices]
    y = y[valid_indices]
    coord = coord[valid_indices]

    print('the size of dataset is:')
    print(X.shape[0])

    if cutoff:
        if direction > 0: ### keep the eastern side
            masklon = coord[:, 1] > cutlon
        else:
            masklon = coord[:, 1] <= cutlon
    
        X = X[masklon]
        y = y[masklon]
        coord = coord[masklon]
    
        print('the size of lon filtered dataset is:')
        print(X.shape[0])
        #print(X)
    
    return X, y, coord

In [98]:
calyears = np.array([1985, 1994, 1999, 2003, 2009, 2013, 2018, 2023])

# Initialize empty arrays to store the data for each year
X_cal = []
y_cal = []
coord_cal = []

# Load data for each year
for calyear in calyears:
    sensorshift1 = 0 if calyear in [2018, 2023] else 1  # Set sensorshift1 based on calyear
    
    X, y, coord = data_prep(f'/Users/bz294/Documents/WhiteZone/Points/KML/{calyear}/', 
                            'median.csv', 0, 0, 0, sensorshift1)
    
    X_cal.append(X.astype(float))
    y_cal.append(y)
    coord_cal.append(coord)  # Coordinates should have shape (n, 2) where n is the number of coordinates

# Convert the data into NumPy arrays for further processing
X_cal = np.array(X_cal, dtype=object)  # Each year's data is stored as an object to preserve shapes
y_cal = np.array(y_cal, dtype=object)
coord_cal = np.array(coord_cal, dtype=object)

# Step 1: Remove duplicate coordinates within each year
for i in range(len(calyears)):
    # Convert the coordinate list into a 2D NumPy array of shape (n, 2) and ensure it's float
    coords = np.array(coord_cal[i], dtype=float)  # Ensure that coordinates are in numeric form
    
    # Check if coordinates are 2D and have the correct shape
    if coords.ndim == 2 and coords.shape[1] == 2:
        # Step 1.1: Find the unique coordinates and their indices
        unique_coords, unique_indices = np.unique(coords, axis=0, return_index=True)
        
        # Round the coordinates to 6 decimal places for consistent precision
        unique_coords = np.round(unique_coords, decimals=6)
        
        # Step 1.2: Use the indices to select unique values
        coord_cal[i] = unique_coords
        X_cal[i] = X_cal[i][unique_indices]
        y_cal[i] = y_cal[i][unique_indices]
        
        print(f"Year {calyears[i]}: {len(coords)} -> {len(unique_coords)} unique coordinates")
    else:
        print(f"Warning: Invalid coordinate shape in year {calyears[i]}")

# Check the results
print("\nSummary of coordinates after removing duplicates:")
for i, calyear in enumerate(calyears):
    print(f"Year {calyear}: {len(coord_cal[i])} coordinates")

training data is:
(1579, 12)
the size of dataset is:
1439
training data is:
(1573, 12)
the size of dataset is:
1442
training data is:
(1577, 12)
the size of dataset is:
1444
training data is:
(1581, 12)
the size of dataset is:
1448
training data is:
(1584, 12)
the size of dataset is:
1452
training data is:
(1586, 12)
the size of dataset is:
1455
training data is:
(1584, 12)
the size of dataset is:
1462
training data is:
(1076, 12)
the size of dataset is:
1061
Year 1985: 1439 -> 1422 unique coordinates
Year 1994: 1442 -> 1432 unique coordinates
Year 1999: 1444 -> 1430 unique coordinates
Year 2003: 1448 -> 1430 unique coordinates
Year 2009: 1452 -> 1431 unique coordinates
Year 2013: 1455 -> 1432 unique coordinates
Year 2018: 1462 -> 1441 unique coordinates
Year 2023: 1061 -> 1041 unique coordinates

Summary of coordinates after removing duplicates:
Year 1985: 1422 coordinates
Year 1994: 1432 coordinates
Year 1999: 1430 coordinates
Year 2003: 1430 coordinates
Year 2009: 1431 coordinates
Y

In [100]:
coord_sets = []
for coords in coord_cal:
    coord_sets.append(set(map(tuple, np.round(coords, decimals=6))))  # Round coordinates to 6 digits

# Step 3: Find the intersection of coordinates across all years (common coordinates)
common_coords = set.intersection(*coord_sets)

# Step 4: Extract indices of the common coordinates for each year
common_indices = []

for coords in coord_cal:
    # Convert the coordinates into tuples for comparison
    coords_tuple = set(map(tuple, np.round(coords, decimals=6)))  # Round coordinates to 6 digits
    
    # Find indices of common coordinates in each year
    indices = [i for i, coord in enumerate(map(tuple, np.round(coords, decimals=6))) if coord in common_coords]
    common_indices.append(indices)

    print(f"Year {calyears[len(common_indices) - 1]}: Found {len(indices)} common coordinates")

# Check the final results
print("\nSummary of common coordinates and indices across all years:")
for i, year in enumerate(calyears):
    print(f"Year {year} has {len(common_indices[i])} common coordinates with others.")



Year 1985: Found 1014 common coordinates
Year 1994: Found 1014 common coordinates
Year 1999: Found 1014 common coordinates
Year 2003: Found 1014 common coordinates
Year 2009: Found 1014 common coordinates
Year 2013: Found 1014 common coordinates
Year 2018: Found 1014 common coordinates
Year 2023: Found 1014 common coordinates

Summary of common coordinates and indices across all years:
Year 1985 has 1014 common coordinates with others.
Year 1994 has 1014 common coordinates with others.
Year 1999 has 1014 common coordinates with others.
Year 2003 has 1014 common coordinates with others.
Year 2009 has 1014 common coordinates with others.
Year 2013 has 1014 common coordinates with others.
Year 2018 has 1014 common coordinates with others.
Year 2023 has 1014 common coordinates with others.


In [105]:
# Get the 39th common index across all years
common_index = 300

# Retrieve the indices for the 39th common coordinate in each year
selected_coords = [coord_cal[i][common_indices[i][common_index]] for i in range(len(calyears))]

# Step 2: Check if all the coordinates are the same across years for the 39th common index
if len(set(map(tuple, selected_coords))) == 1:
    print(f"Coordinates for the 39th common index are the same across all 8 years: {selected_coords[0]}")
else:
    print(f"Coordinates for the 39th common index are inconsistent across years:")
    for i, year in enumerate(calyears):
        print(f"Year {year}: {selected_coords[i]}")

Coordinates for the 39th common index are the same across all 8 years: [ 25.243973 -80.487025]


In [109]:
# Step 1: Initialize a list to store the common indices where y_cal values are the same across all years
same_y_cal_indices = []

# Step 2: Iterate over each common index
for common_index in range(len(common_indices[0])):  # Assuming all years have the same number of common indices
    # Step 2.1: Check the y_cal values for the current common index across all years
    y_values = [y_cal[i][common_indices[i][common_index]] for i in range(len(calyears))]
    
    # Step 2.2: If all y_cal values are the same, add the common index to the list
    if len(set(y_values)) == 1:
        same_y_cal_indices.append(common_index)

# Step 3: Output the results
print(f"Indices with the same y_cal values across all 8 years: {same_y_cal_indices}")

Indices with the same y_cal values across all 8 years: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 59, 61, 62, 63, 64, 65, 66, 67, 68, 71, 73, 75, 76, 80, 81, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 94, 95, 96, 97, 98, 102, 103, 104, 105, 106, 107, 108, 113, 115, 116, 122, 123, 125, 130, 133, 134, 144, 146, 147, 149, 151, 152, 154, 160, 161, 165, 166, 167, 170, 174, 182, 183, 184, 186, 187, 190, 191, 194, 197, 198, 200, 201, 202, 203, 211, 215, 216, 220, 222, 223, 234, 235, 239, 243, 246, 248, 249, 256, 261, 263, 264, 266, 271, 273, 274, 275, 277, 278, 279, 280, 283, 284, 286, 289, 290, 293, 297, 299, 300, 301, 305, 307, 308, 311, 312, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 325, 328, 330, 331, 334, 336, 337, 339, 342, 344, 345, 347, 350, 355, 356, 357, 358, 359, 360, 361, 371, 372, 373, 380, 384, 385, 3

In [123]:
specific_year_index = 0  # Change this index to the desired year (e.g., 0 for the first year)
coord_cal_values_for_year = [coord_cal[specific_year_index][common_indices[specific_year_index][index]] for index in same_y_cal_indices]

# Print the coord_cal values for the specified year
print(f"coord_cal values for year {calyears[specific_year_index]} and indices with the same y_cal values across all years: {len(coord_cal_values_for_year)}")
y_cal_values_for_year = [y_cal[specific_year_index][common_indices[specific_year_index][index]] for index in same_y_cal_indices]

coord_cal values for year 1985 and indices with the same y_cal values across all years: 488


In [126]:
coord_cal_values_for_year_np = np.array(coord_cal_values_for_year)
y_cal_values_for_year_np = np.array(y_cal_values_for_year)
combined_array = np.column_stack((y_cal_values_for_year_np, coord_cal_values_for_year_np))

# Save the combined array to a .npy file
output_filename = 'combined_values.npy'
np.save(output_filename, combined_array)

# Print a confirmation message
print(f"Combined array saved to {output_filename}")

Combined array saved to combined_values.npy
